# Notebook 00 — Data Validation and Preprocessing

**Purpose:** Load the raw arm-level LLM extraction (`copd.csv`), validate its
structure and content, produce cleaned `arms.csv` and `trials.csv` for
downstream notebooks, and merge country/year metadata.

**Data Flow:**

| Direction | File | From / To |
| :--------- | :---- | :--------- |
| Input | `data/misc/copd.csv` | Raw LLM extraction (sibling to JSONL in `data/raw/`) |
| Output | `data/processed/arms.csv` | Feeds all downstream notebooks |
| Output | `data/processed/trials.csv` | Feeds all downstream notebooks |
| Output | `data/processed/country_year.csv` | Feeds downstream notebooks |
| Output | `data/processed/data_dictionary.md` | Reference only |

**Prerequisite notebooks:** None (entry point).

**Index:**

| Section | What it does |
| :------- | :------------ |
| 0 | Setup — paths, imports, schema hash |
| 1 | Load and validate arm-level data (column counts, NA flags) |
| 2 | Validate categorical fields (arm labels, diagnosis, healthcare setting) |
| 3 | Structured-array field audit (key counts, completeness) |
| 4 | Validate arm counts per trial (2–3 arms expected) |
| 5 | Load and clean country/year lookup |
| 6 | Build trial-level dataset (arm aggregation + exclusion audit) |
| 7 | Save processed datasets |
| 8 | Generate data dictionary |
| 9 | Summary and sanity checklist |

> **Shared code:** All notebooks import from `src/data_loading.py` — the single
> entry point for file paths, column schemas, and validation rules. N01
> additionally uses `src/loaders.py`, `src/normalization.py`, and
> `src/agreement.py`. N02 shares those plus `src/agreement.py`. N03 uses
> `src/aggregation.py`, `src/statistics.py`, `src/plotting.py`,
> `src/geography.py`, and build scripts in `scripts/`.

**Notebook-specific notes:**

- **Arms vs trials:** Each row in `arms.csv` is one study *arm* (typically 2 per
  trial: control + intervention). `trials.csv` aggregates arms to *trial* level
  using n-weighted means. Downstream notebooks should use trial-level data for
  frequencies and regressions; arm counts are provided for transparency.
- **Input requirements:** The input CSV must match `src/data_loading.py`
  expectations (cov_nr, arm, n, age_mean, age_sd, gender_pct_female,
  fev1_pct_mean, fev1_pct_sd, bmi_mean, bmi_sd, healthcare_setting).
  `load_arms()` validates row counts, unique trials, required columns, and
  runs type-checks. Structured-array fields are parsed downstream.
- **Exclusion logic:** One trial ({5108}) is excluded automatically by
  `load_arms()` because >30% of participants did not have COPD (the trial
  enrolled mixed-condition patients; the COPD subgroup is too small for
  reliable evidence mapping).


## Section 0: Setup

In [1]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# ROOT resolves to the project directory whether this notebook runs from
# inside notebooks/ or from the project root. If the project is cloned to a
# different machine, ensure notebooks/ and src/ remain siblings.
ROOT = Path().resolve().parent if Path().resolve().name == "notebooks"          else Path().resolve()
sys.path.insert(0, str(ROOT))

# ── External libraries ───────────────────────────────────────────────────────
import pandas as pd

# ── Project modules (src/) ───────────────────────────────────────────────────
#   src/data_loading — load, validate, aggregate: load_arms, create_trials,
#       load_and_clean_country_year, merge_year_country, parse_structured_array,
#       STRUCTURED_ARRAY_FIELDS, value_has_data, get_schema_hash, path constants

from src.analysis.data_loading import (
    ROOT, DATA_PROCESSED, ARMS_PATH, TRIALS_PATH, COUNTRY_YEAR_PATH,
    COPD_CSV_PATH, COUNTRY_YEAR_RAW_PATH,
)
from src.analysis.data_loading import load_arms
from src.analysis.data_loading import create_trials
from src.analysis.data_loading import load_and_clean_country_year
from src.analysis.data_loading import merge_year_country
from src.analysis.data_loading import STRUCTURED_ARRAY_FIELDS, parse_structured_array, value_has_data
from src.analysis.data_loading import get_schema_hash
#     column names, the hash changes — downstream can detect stale data by
#     comparing hashes. Used with get_schema_hash().

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Schema hash: {get_schema_hash()}")


Project root: /mnt/data_ssd/thesis-writing
Schema hash: 2b702021


## Section 1: Load and validate arm-level data

Load `copd.csv` and run structural sanity checks:
- 65 trials in raw CSV → 64 after excluding trial 5108 (>30% non-COPD)
- 130–140 total arms
- Required columns present
- Data types correct

**Note:** This is an arm-level validation — each row is one study arm.


In [2]:
# Load and validate arm-level data from raw CSV. Expected: ~64 unique trials,
# 128–142 rows, 2–3 arms per trial. Prints column-level NA counts per field.
arms = load_arms(COPD_CSV_PATH)
print(f"Shape: {arms.shape}")
print(f"Unique trials: {arms['cov_nr'].nunique()}")
print(f"Arms per trial: {arms.groupby('cov_nr').size().describe()}")
print()
print("Columns:")
for c in arms.columns:
    na = arms[c].isna().sum()
    na_pct = na / len(arms) * 100
    missingness_marker = " ⚠️" if na_pct > 50 else ""
    print(f"  {c:<45} {arms[c].dtype!s:<10} {na:>4}/{len(arms)} NA ({na_pct:5.1f}%){missingness_marker}")


Shape: (134, 71)
Unique trials: 64
Arms per trial: count    64.000000
mean      2.093750
std       0.293785
min       2.000000
25%       2.000000
50%       2.000000
75%       2.000000
max       3.000000
dtype: float64

Columns:
  cov_nr                                        int64         0/134 NA (  0.0%)
  arm                                           str           0/134 NA (  0.0%)
  arm_explanation                               str           0/134 NA (  0.0%)
  needs_discussion_arm                          bool          0/134 NA (  0.0%)
  needs_discussion_arm_explanation              str         132/134 NA ( 98.5%) ⚠️
  n                                             int64         0/134 NA (  0.0%)
  time_intervention_days                        float64       6/134 NA (  4.5%)
  time_followup_days                            float64       4/134 NA (  3.0%)
  time_total_days                               float64       4/134 NA (  3.0%)
  needs_discussion_time                         b

## Section 2: Validate categorical fields

Diagnosis is reported in structured-array format. N03 recodes to `COPD only` / `COPD + other`.
(arm-level check)


In [3]:
# Check categorical fields for expected distributions. All arms should be
# treat1/treat2/control; diagnosis should be predominantly COPD.
for col in ['arm', 'diagnosis', 'healthcare_setting']:
    if col not in arms.columns:
        continue  # skip silently if renamed upstream; output will have gaps
    print(arms[col].value_counts(dropna=False).to_string())
    print()

print("healthcare_setting labels:")
print(arms['healthcare_setting_label'].value_counts(dropna=False).to_string())


arm
treat1     64
control    64
treat2      6

diagnosis
COPD                                                                              130
COPD~CHF                                                                            2
COPD: 50 (70.42%)~ILD: 5 (7.04%)~Bronchiectasis: 10 (14.08%)~Asthma: 6 (8.45%)      1
COPD: 50 (70.42%)~ILD: 6 (8.45%)~Bronchiectasis: 9 (12.68%)~Asthma: 6 (8.45%)       1

healthcare_setting
2    105
3     15
1     14

healthcare_setting labels:
healthcare_setting_label
Secondary    105
Community     15
Primary       14


## Section 3: Check structured-array fields

Count how many arms have data in each structured-array field.

In [4]:
# Structured-array fields store 'Key: Value~Key: Value' strings.
# Count how many arms carry actual data in each to assess reporting
# completeness at arm level.
for col in STRUCTURED_ARRAY_FIELDS:
    if col not in arms.columns:
        continue  # skip silently if renamed upstream; output will have gaps
    n_non_na = arms[col].notna().sum()
    n_with_keys = sum(1 for v in arms[col] if value_has_data(v))
    keys = set()
    for v in arms[col].dropna():
        parsed = parse_structured_array(v)
        if parsed:
            keys.update(k for k, _ in parsed)
    print(f"  {col:<40} {n_non_na:>3}/{len(arms):<3} arms ({n_non_na/len(arms)*100:>6.1f}%)  "
          f"non-empty: {n_with_keys:>3}  unique keys: {len(keys):>3}")


  diagnosis                                134/134 arms ( 100.0%)  non-empty: 134  unique keys:   5
  smoking_status                            77/134 arms (  57.5%)  non-empty:  77  unique keys:  22
  ses_income                                12/134 arms (   9.0%)  non-empty:  12  unique keys:  11
  ses_living_situation                      37/134 arms (  27.6%)  non-empty:  37  unique keys:  21
  ses_relationship_status                   19/134 arms (  14.2%)  non-empty:  19  unique keys:  11
  ses_job_status                            24/134 arms (  17.9%)  non-empty:  24  unique keys:   9
  ses_living_location                        8/134 arms (   6.0%)  non-empty:   8  unique keys:   3
  educational_level                         38/134 arms (  28.4%)  non-empty:  38  unique keys:  36
  ethnicity                                 23/134 arms (  17.2%)  non-empty:  23  unique keys:  18
  digital_literacy_possession               13/134 arms (   9.7%)  non-empty:  13  unique keys:   9


### Unique keys per structured-array field

In [5]:
# Print all unique keys per structured-array field (no limit).
for col in STRUCTURED_ARRAY_FIELDS:
    keys = set()
    for v in arms[col].dropna():
        parsed = parse_structured_array(v)
        if parsed:
            keys.update(k.strip() for k, _ in parsed if k.strip())
    if keys:
        print(f"{col} ({len(keys)} unique keys):")
        for k in sorted(keys):
            print(f"  {k}")
        print()
    else:
        print(f"{col}: no keys found")


diagnosis (5 unique keys):
  Asthma
  Bronchiectasis
  CHF
  COPD
  ILD

smoking_status (22 unique keys):
  Active
  Active smokers
  Current
  Current or previous smokers
  Current smoker
  Currently smoking
  Ex smoker
  Ex-smoker
  Ex-smoker (<2 years)
  Ex-smoker (≥2 years)
  Former
  Former smoker
  Missing
  Never
  Never smoked
  Never smoker
  Non-smokers
  Nonsmokers
  Past
  Smoker
  Smokers
  non-smoking (inferred)

ses_income (11 unique keys):
  $10,000 - $19,999/yr
  $20,000 - $39,999/yr
  < $1000
  <$10,000
  >$40,000/yr
  Annual income < $30k
  Annual income <€6000
  Financial insecurity
  Income ≤$30,000
  Yes
  ≥$1000

ses_living_situation (21 unique keys):
  Alone
  Family
  Live with spouse or other
  Lived alone
  Lived with others
  Lives alone
  Living alone
  Living with partner
  Living with spouse
  Married or living with a partner
  Married or living with someone
  Other situation
  Solo
  Supported accommodation
  With family
  With friends
  With other relat

## Section 4: Validate arm counts per trial

Each trial should have 2–3 arms (typically 2).

In [6]:
# Verify arm counts per trial: 2 is expected, 3 indicates multi-arm.
# Trials with <2 arms would be malformed.
arm_counts = arms.groupby('cov_nr').size()
print("Arms per trial distribution:")
print(arm_counts.describe())
print()
print("Trials with >2 arms:")
multi_arm_trials = arm_counts[arm_counts > 2]
if len(multi_arm_trials) > 0:
    for cov, count in multi_arm_trials.items():
        print(f"  cov_nr={cov}: {count} arms — {arms[arms.cov_nr==cov]['arm'].tolist()}")

print()
print("Trials with <2 arms:")
malformed_trials = arm_counts[arm_counts < 2]
if len(malformed_trials) > 0:
    for cov, count in malformed_trials.items():
        print(f"  cov_nr={cov}: {count} arms — REVIEW NEEDED")


Arms per trial distribution:
count    64.000000
mean      2.093750
std       0.293785
min       2.000000
25%       2.000000
50%       2.000000
75%       2.000000
max       3.000000
dtype: float64

Trials with >2 arms:
  cov_nr=3089: 3 arms — ['treat1', 'treat2', 'control']
  cov_nr=4104: 3 arms — ['treat1', 'treat2', 'control']
  cov_nr=4876: 3 arms — ['treat1', 'treat2', 'control']
  cov_nr=5232: 3 arms — ['treat1', 'treat2', 'control']
  cov_nr=5660: 3 arms — ['treat1', 'treat2', 'control']
  cov_nr=6159: 3 arms — ['treat1', 'treat2', 'control']

Trials with <2 arms:


## Section 5: Load and clean country/year lookup

In [7]:
# Load and clean manual country/year lookup. Merge for temporal/geographic
# analysis downstream.
country_year_df = load_and_clean_country_year(COUNTRY_YEAR_RAW_PATH)
print(f"Country/year records: {len(country_year_df)}")
print(f"Year range: {country_year_df['publication_year'].min()}-{country_year_df['publication_year'].max()}")
print(f"Unique countries (first 15):")
all_countries = country_year_df['country'].unique()
for c in sorted(all_countries)[:15]:
    print(f"  {c}")

# Save cleaned version
country_year_df.to_csv(COUNTRY_YEAR_PATH, index=False)
print(f"\nSaved to {COUNTRY_YEAR_PATH}")


Country/year records: 65
Year range: 2006-2023
Unique countries (first 15):
  Australia
  Belgium
  Belgium and Spain
  Canada
  China
  Denmark
  Europe (Belgium, Greece, UK, Switzerland, Netherlands)
  France, Germany, Italy, Spain
  Germany
  Greece
  Italy
  Korea
  Netherlands
  New Zealand
  Norway

Saved to /mnt/data_ssd/thesis-writing/data/processed/country_year.csv


## Section 6: Build trial-level dataset

Aggregate arm-level → trial-level using n-weighted means for continuous
variables. Trial 5108 was excluded by `load_arms()` (>30% non-COPD
participants) and is not present in the `arms` DataFrame.

**Healthcare setting:** If all arms within a trial share the same setting,
the trial gets that single label. If arms disagree (genuine trial design —
e.g. intervention in community, control in primary care), the trial is
labeled `"Mixed: A / B"` with ordered by ordinal code (1=Primary, 2=Secondary, 3=Community) setting names. This
replaces the previous silent modal aggregation that biased toward lower
numeric codes.


In [8]:
# Trial 5108 was excluded by load_arms() — >30% non-COPD participants.
# The 64 trials below form the final corpus.
print("Exclusion audit:")
print("  cov_nr=5108: >30% non-COPD participants (excluded by load_arms)")
print(f"  Trials after exclusion: {arms['cov_nr'].nunique()}")

# Aggregate to trial-level (n-weighted means + healthcare "Mixed" logic)
trials = create_trials(arms)
print(f"\nTrials: {len(trials)}")

# Merge country/year metadata
trials = merge_year_country(trials)

missing_year = trials['publication_year'].isna().sum()
missing_country = trials['country'].isna().sum()
print(f"Trials with missing year: {missing_year}")
print(f"Trials with missing country: {missing_country}")

print(f"\nTrials summary:")
print(f"  Total N across all trials: {trials['total_n'].sum():.0f}")
print(f"  Median trial N: {trials['total_n'].median():.0f}")
print(f"  Year range: {trials['publication_year'].min():.0f}-{trials['publication_year'].max():.0f}")

print(f"\nHealthcare setting distribution (trial-level):")
print(trials['healthcare_setting_label'].value_counts().to_string())


Exclusion audit:
  cov_nr=5108: >30% non-COPD participants (excluded by load_arms)
  Trials after exclusion: 64



Trials: 64
Trials with missing year: 0
Trials with missing country: 0

Trials summary:
  Total N across all trials: 13953
  Median trial N: 106
  Year range: 2006-2023

Healthcare setting distribution (trial-level):
healthcare_setting_label
Secondary                       47
Community                        4
Mixed: Primary / Community       4
Primary                          4
Mixed: Secondary / Community     3
Mixed: Primary / Secondary       2


## Section 7: Save processed datasets

In [9]:
# Persist validated data. Downstream notebooks read from these CSVs.
# Save arms
arms.to_csv(ARMS_PATH, index=False)
print(f"Arms saved: {ARMS_PATH} ({len(arms)} rows)")

# Save trials
trials.to_csv(TRIALS_PATH, index=False)
print(f"Trials saved: {TRIALS_PATH} ({len(trials)} rows)")

# Save schema version missingness_marker
schema_file = DATA_PROCESSED / "schema_version.txt"
schema_file.write_text(f"schema_hash: {get_schema_hash()}\ndate: {pd.Timestamp.now().isoformat()}\n")

print(f"\nProcessed data written. Schema version: {get_schema_hash()}")


Arms saved: /mnt/data_ssd/thesis-writing/data/processed/arms.csv (134 rows)
Trials saved: /mnt/data_ssd/thesis-writing/data/processed/trials.csv (64 rows)

Processed data written. Schema version: 2b702021


## Section 8: Data dictionary

Document each field for reference.
Healthcare setting is aggregated from arm-level using the 'Mixed: A / B' logic described in Section 6.


In [10]:
# Generate a human-readable data dictionary of every field in both datasets.
dictionary_lines = [
    "# Data Dictionary — COPD Evidence Map",
    "",
    f"Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}",
    f"Schema hash: {get_schema_hash()}",
    "",
    "## Arm-level fields (arms.csv)",
    "",
    "| Field | Type | Description |",
    "|-------|------|-------------|",
]

field_descriptions = {
    'cov_nr': ('int', 'Unique trial identifier'),
    'arm': ('str', 'Arm label (treat1, treat2, control)'),
    'arm_explanation': ('str', 'Human-readable arm description'),
    'n': ('int', 'Number of participants in arm'),
    'time_intervention_days': ('int', 'Duration of intervention phase (days)'),
    'time_followup_days': ('int', 'Duration of follow-up after intervention (days)'),
    'time_total_days': ('int', 'Total study duration (days)'),
    'diagnosis': ('structured_array', 'COPD diagnosis/subtypes'),
    'gender_pct_female': ('float', 'Percentage female participants'),
    'age_mean': ('float', 'Mean age (years)'),
    'age_sd': ('float', 'SD of age'),
    'bmi_mean': ('float', 'Mean BMI (kg/m²)'),
    'bmi_sd': ('float', 'SD of BMI'),
    'smoking_status': ('structured_array', 'Smoking status distribution'),
    'fev1_pct_mean': ('float', 'Mean FEV1% predicted'),
    'fev1_pct_sd': ('float', 'SD of FEV1% predicted'),
    'disease_severity_other': ('structured_array', 'Other disease severity measures'),
    'healthcare_setting': ('int', 'Setting: 1=primary, 2=secondary, 3=community/home'),
    'healthcare_setting_label': ('str', 'Human-readable setting label'),
    'health_literacy': ('bool', 'Was health literacy reported?'),
    'digital_literacy': ('bool', 'Was digital literacy reported?'),
    'digital_strategy_excludes': ('int', 'Excludes participants based on digital criteria'),
    'digital_strategy_provides_equipment': ('int', 'Provides equipment to participants'),
    'digital_strategy_provides_training': ('int', 'Provides training on digital tools'),
    'digital_strategy_provides_ongoing_support': ('int', 'Provides ongoing technical support'),
    'digital_literacy_possession': ('str', 'Digital device possession (free text)'),
    'digital_literacy_frequency': ('str', 'Frequency of digital tool use (free text)'),
    'digital_literacy_skills': ('str', 'Self-reported digital skills (free text)'),
    
    'ses_income': ('structured_array', 'Income distribution'),
    'ses_living_situation': ('structured_array', 'Living situation distribution'),
    'ses_relationship_status': ('structured_array', 'Relationship status distribution'),
    'ses_job_status': ('structured_array', 'Employment status distribution'),
    'ses_living_location': ('structured_array', 'Living location distribution'),
    'educational_level': ('structured_array', 'Educational level distribution'),
    'ethnicity': ('structured_array', 'Ethnicity distribution'),
}

for col in sorted(arms.columns):
    if col.startswith('needs_discussion_') or col.startswith('_metadata'):
        continue
    dtype, desc = field_descriptions.get(col, (str(arms[col].dtype), ''))
    dictionary_lines.append(f"| `{col}` | {dtype} | {desc} |")

dictionary_lines.extend([
    "",
    "## Trial-level fields (trials.csv)",
    "",
    "| Field | Type | Description |",
    "|-------|------|-------------|",
    "| cov_nr | int | Unique trial identifier |",
    "| n_arms | int | Number of arms in trial |",
    "| total_n | int | Total participants across arms |",
    "| age_mean | float | N-weighted mean age across arms |",
    "| fev1_pct_mean | float | N-weighted mean FEV1% predicted |",
    "| bmi_mean | float | N-weighted mean BMI |",
    "| gender_pct_female | float | N-weighted mean % female |",
    "| healthcare_setting | int | Modal healthcare setting |",
    "| healthcare_setting_label | str | Human-readable setting label |",
    "| publication_year | int | Year of publication |",
    "| country | str | Country/countries of trial |",
])

data_dict_path = DATA_PROCESSED / "data_dictionary.md"
data_dict_path.write_text("\n".join(dictionary_lines))
print(f"Data dictionary saved to {data_dict_path}")


Data dictionary saved to /mnt/data_ssd/thesis-writing/data/processed/data_dictionary.md


## Section 9: Summary

Sanity checklist for downstream notebooks:
- ✓ 64 trials in final corpus (1 excluded: 5108, >30% non-COPD)
- ✓ Healthcare setting labels assigned (9 trials with mixed settings)
- ✓ Country/year merged
- ✓ All downstream notebooks read from `data/processed/`


In [11]:
print("=== Notebook 00 complete ===")
print(f"Completed at: {pd.Timestamp.now().isoformat()}")
print(f"Schema hash: {get_schema_hash()}")

print(f"Schema hash: 2b702021 (this revision's reference)")
# Schema staleness check: compare computed hash against stored version.
# If column schemas changed since last run, downstream notebooks may
# produce stale results.
current_hash = get_schema_hash()
version_file = ROOT / "data" / "processed" / "schema_version.txt"
if version_file.exists():
    stored_hash = version_file.read_text().strip()
    if current_hash != stored_hash:
        print(f"WARNING: Schema hash changed from {stored_hash} to {current_hash} — downstream data may be stale")
    else:
        print(f"Schema hash verified: {current_hash}")
else:
    print(f"Schema hash: {current_hash} (first run — no prior hash stored)")
version_file.write_text(current_hash)


=== Notebook 00 complete ===
Completed at: 2026-06-10T00:59:05.268451
Schema hash: 2b702021
Schema hash: 2b702021 (this revision's reference)
date: 2026-06-10T00:59:05.234793 to 2b702021 — downstream data may be stale


8